# Optional one-time private dataset builder

Settings: **GPU T4 x2**, **Internet ON**. Run cells 1?3 once, or Save & Run All.
This compiles pinned SM75 CUDA source and bundles its complete engine libraries,
then optionally downloads the pinned GGUF. It makes no generation requests.
Building here consumes Kaggle session time; a CPU-only CUDA build is also documented
in `knowledgebase/KAGGLE_FAST_START.md`. No fixed build/startup time is promised.

The output `freecompute-assets/` contains `engine/`, optional `models/`, and
`freecompute-assets.json` with source pin, build flags, platform, minimum glibc,
engine version, file SHA256 hashes and pinned model metadata.


In [ ]:
# 1. Pinned assets (the server notebook uses these same defaults)
BUILD_CONFIG = {
    "REPO_ID": "huihui-ai/Huihui-Qwen3.8-27B-abliterated-GGUF",
    "FILENAME": "Huihui-Qwen3.8-27B-abliterated-UD-DW-Q4_K_M.gguf",
    "REVISION": "3f101cd22b7999228bbd5d79a33975414eb9758b",
    "LLAMA_COMMIT": "2b129ccfa03aea330d2d9ac4650a10de393dbe3a",
}
DOWNLOAD_MODEL = True  # Optional advanced choice: False produces engine-only assets.


In [ ]:
# 2. Pinned engine and matching runtime libraries
"""Self-contained dataset helpers embedded in both notebooks. No network on import."""
import hashlib
import json
import os
from pathlib import Path, PurePosixPath
import platform
import re
import shutil
import subprocess

BUILD_FLAGS = ['GGML_CUDA=ON', 'GGML_CUDA_NO_VMM=ON', 'CMAKE_CUDA_ARCHITECTURES=75',
               'CMAKE_BUILD_TYPE=Release', 'LLAMA_BUILD_TESTS=OFF', 'LLAMA_CURL=OFF', 'GGML_NATIVE=OFF']
CUDA_LIBRARIES = ('libcudart.so.12', 'libcublas.so.12', 'libcublasLt.so.12')


def asset_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def asset_path(root, name):
    relative = PurePosixPath(name)
    if not name or relative.is_absolute() or '..' in relative.parts or '\\' in name or ':' in name:
        raise ValueError('Unsafe dataset manifest path')
    path = Path(root).joinpath(*relative.parts)
    # No symlinked executables, libraries, parents or models from input datasets.
    for part in (path, *path.parents):
        if part == Path(root).parent:
            break
        if part.is_symlink():
            raise ValueError('Dataset assets must not be symlinks')
    if not path.resolve().is_relative_to(Path(root).resolve()):
        raise ValueError('Dataset path escapes its root')
    return path


def verify_asset_manifest(manifest_path, config):
    manifest_path = Path(manifest_path)
    if manifest_path.is_symlink():
        raise ValueError('Dataset manifest must not be a symlink')
    data = json.loads(manifest_path.read_text(encoding='utf-8'))
    if not isinstance(data, dict):
        raise ValueError('Dataset manifest must be a mapping')
    engine = data.get('engine', {})
    if not isinstance(engine, dict) or data.get('schema_version') != 1 or engine.get('commit') != config['LLAMA_COMMIT'] or engine.get('cuda_arch') != '75' or engine.get('build_flags') != BUILD_FLAGS:
        raise ValueError('Dataset engine does not match the pinned T4 build')
    libc, version = platform.libc_ver()
    required = engine.get('minimum_glibc', '')
    if libc != 'glibc' or not re.fullmatch(r'\d+\.\d+', required) or tuple(map(int, version.split('.')[:2])) < tuple(map(int, required.split('.'))):
        raise ValueError('Dataset engine requires a newer/different glibc runtime')
    if engine.get('platform') != 'Linux-x86_64':
        raise ValueError('Dataset engine platform is unsupported')
    files = engine.get('files', {})
    server = engine.get('server', '')
    if not isinstance(files, dict) or not isinstance(server, str) or not server.startswith('engine/') or server not in files or not files:
        raise ValueError('Dataset engine manifest has no server/checksums')
    root = manifest_path.parent
    for name, digest in files.items():
        if not isinstance(name, str) or not name.startswith('engine/') or len(PurePosixPath(name).parts) != 2 or not isinstance(digest, str) or not re.fullmatch(r'[0-9a-f]{64}', digest):
            raise ValueError('Invalid engine file checksum declaration')
        path = asset_path(root, name)
        if not path.is_file() or asset_sha256(path) != digest:
            raise ValueError('Dataset engine file is missing or has changed: ' + name)
    expected = {str(p.relative_to(root)).replace('\\', '/') for p in (root / 'engine').rglob('*') if p.is_file()}
    if expected != set(files):
        raise ValueError('Dataset engine contains undeclared files')
    for library in (*CUDA_LIBRARIES, 'libggml-cuda.so'):
        if 'engine/' + library not in files:
            raise ValueError('Dataset engine is missing a required runtime library')
    model_path = None
    model = data.get('model')
    if model and not isinstance(model, dict):
        raise ValueError('Invalid model manifest')
    if model:
        if (model.get('repo'), model.get('revision'), model.get('filename')) == (config['REPO_ID'], config['REVISION'], config['FILENAME']):
            model_path = asset_path(root, model.get('path', ''))
            if not re.fullmatch(r'[0-9a-f]{64}', model.get('sha256', '')) or not model_path.is_file() or model_path.stat().st_size != model.get('bytes') or asset_sha256(model_path) != model['sha256']:
                raise ValueError('Dataset model checksum/size verification failed')
    return data, asset_path(root, server), model_path


def find_cached_assets(input_dir, scratch, config, report=print):
    for path in sorted(Path(input_dir).glob('**/freecompute-assets.json')):
        try:
            data, server, model = verify_asset_manifest(path, config)
            destination = Path(scratch) / 'cached-engine'
            destination.mkdir(parents=True, exist_ok=True)
            for name in data['engine']['files']:
                source = asset_path(path.parent, name)
                target = destination / source.name
                shutil.copyfile(source, target)
                if asset_sha256(target) != data['engine']['files'][name]:
                    raise ValueError('Copied engine checksum verification failed')
            copied_server = destination / server.name
            copied_server.chmod(0o755)
            report('FAST START: verified private dataset engine; no download/build.')
            return copied_server, model, dict(data['engine'], distribution='user-attached dataset; manifest/checksums verified')
        except (ValueError, OSError, TypeError, KeyError) as exc:
            report('Skipping incompatible dataset: ' + str(exc))
    return None, None, None


def build_source_engine(config, scratch, destination=None, report=print):
    source = Path(scratch) / 'llama.cpp'
    if not source.exists():
        subprocess.run(['git', 'clone', '--no-checkout', '--depth', '1', 'https://github.com/ggml-org/llama.cpp.git', str(source)], check=True)
    subprocess.run(['git', '-C', str(source), 'fetch', '--depth', '1', 'origin', config['LLAMA_COMMIT']], check=True)
    subprocess.run(['git', '-C', str(source), 'checkout', '--detach', config['LLAMA_COMMIT']], check=True)
    commit = subprocess.check_output(['git', '-C', str(source), 'rev-parse', 'HEAD'], text=True).strip()
    if commit != config['LLAMA_COMMIT']:
        raise RuntimeError('Source checkout does not match the configured engine pin')
    if subprocess.check_output(['git', '-C', str(source), 'status', '--porcelain'], text=True).strip():
        raise RuntimeError('Build checkout has local changes; refusing to call it the pinned source')
    build = source / 'build'
    subprocess.run(['cmake', '-S', str(source), '-B', str(build), *['-D' + flag for flag in BUILD_FLAGS]], check=True)
    subprocess.run(['cmake', '--build', str(build), '--target', 'llama-server', '--parallel', '2'], check=True)
    destination = Path(destination or Path(scratch) / 'source-engine')
    destination.mkdir(parents=True, exist_ok=True)
    for file in (build / 'bin').iterdir():
        if file.is_file() and (file.name == 'llama-server' or '.so' in file.name):
            shutil.copyfile(file, destination / file.name)
    for library in CUDA_LIBRARIES:
        paths = list(Path('/usr/local/cuda').glob('**/' + library))
        if not paths:
            raise RuntimeError('CUDA toolkit is missing ' + library)
        shutil.copyfile(paths[0], destination / library)
    server = destination / 'llama-server'
    if not server.is_file():
        raise RuntimeError('Pinned build did not produce llama-server')
    server.chmod(0o755)
    os.environ['LD_LIBRARY_PATH'] = str(destination) + ':' + os.environ.get('LD_LIBRARY_PATH', '')
    version = subprocess.check_output([str(server), '--version'], stderr=subprocess.STDOUT, text=True).strip()
    if not re.search(r'\b' + commit[:7] + r'[0-9a-f]*\b', version):
        raise RuntimeError('Built engine does not report the pinned version')
    provenance = {'commit': commit, 'cuda_arch': '75', 'build_flags': BUILD_FLAGS,
                  'minimum_glibc': '.'.join(platform.libc_ver()[1].split('.')[:2]), 'platform': 'Linux-x86_64',
                  'version_observed': version, 'server': 'engine/llama-server',
                  'files': {'engine/' + file.name: asset_sha256(file) for file in sorted(destination.iterdir()) if file.is_file()},
                  'distribution': 'pinned source build on this runtime'}
    report('Pinned T4 engine built; save it once as a private dataset to avoid future builds.')
    return server, provenance

WORK = Path('/kaggle/working/freecompute-assets')
SCRATCH = Path('/kaggle/tmp/build_dataset')
WORK.mkdir(parents=True, exist_ok=True)
SCRATCH.mkdir(parents=True, exist_ok=True)
server, engine_provenance = build_source_engine(BUILD_CONFIG, SCRATCH, WORK / 'engine')
manifest = {'schema_version': 1, 'engine': engine_provenance, 'model': None}
(WORK / 'freecompute-assets.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print('Engine-only manifest saved; Cell 3 optionally adds the model.')


In [ ]:
# 3. Optional pinned model; seal dataset metadata
import sys
if DOWNLOAD_MODEL:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub>=0.34,<1.0', 'hf_xet>=1.0'], check=True)
    from huggingface_hub import hf_hub_download
    model = Path(hf_hub_download(repo_id=BUILD_CONFIG['REPO_ID'], filename=BUILD_CONFIG['FILENAME'], revision=BUILD_CONFIG['REVISION'], local_dir=str(WORK / 'models')))
    manifest['model'] = {'repo': BUILD_CONFIG['REPO_ID'], 'revision': BUILD_CONFIG['REVISION'], 'filename': BUILD_CONFIG['FILENAME'],
                         'path': str(model.relative_to(WORK)), 'bytes': model.stat().st_size, 'sha256': asset_sha256(model)}
(WORK / 'freecompute-assets.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
verify_asset_manifest(WORK / 'freecompute-assets.json', BUILD_CONFIG)
print('DATASET VERIFIED. Save the entire freecompute-assets directory as a PRIVATE dataset.')


## Save and reuse

1. Save Version ? Save & Run All. After completion, use Output ? New Dataset.
2. Keep the dataset **private**. Include the entire `freecompute-assets` directory,
   including every library and the JSON manifest. Do not include notebook logs/keys.
3. In **freecompute_dual_gpu_server.ipynb**, Add Input ? your private dataset.
4. Run cells 1?8. Cell 3 reports verified FAST START; Cell 4 still checks the engine
   version and both T4 devices. No model generation occurs during these checks.

The manifest is an integrity/provenance record for your trusted dataset, not a signature
that makes a third-party executable safe. Never attach an unknown author's engine.
